# Stage 2B - Expected Price + BE Export

Predicts `expprice` / `expstd` for every player in `output_rp.csv` using the per-role
models trained by Stage 2A, runs the residual-correction experiment (D3), and exports the
BE-ready `data/final/{SEASON}/players.csv` (14 columns, exact BE contract names).

The legacy multi-feature experiments (GPR / Ridge / Lasso / SVM, old cells 12-45) were
moved to `explorations/legacy_regressor_experiments.ipynb`.


In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

# Resolve repo root (notebook may run from pipeline/ or repo root)
BASE = os.getcwd()
if os.path.basename(BASE) == 'pipeline':
    BASE = os.path.abspath(os.path.join(BASE, '..'))
os.chdir(BASE)
sys.path.insert(0, BASE)

# Load centralized config (shared with Stage 2A)
with open('pipeline/config/stage2a2b.yaml') as _f:
    CFG = yaml.safe_load(_f)

SEASON = "25-26"
INTER_DIR = f"data/intermediate/{SEASON}"
MODELS_DIR = f"models/fvm_distribution/{SEASON}"
FINAL_DIR = f"data/final/{SEASON}"
Path(FINAL_DIR).mkdir(parents=True, exist_ok=True)

print(f"SEASON = {SEASON}")
print(f"MODELS_DIR = {MODELS_DIR}")
print(f"FINAL_DIR  = {FINAL_DIR}")


In [ ]:
# Load Stage 1 output
df = pd.read_csv(Path(INTER_DIR) / "output_rp.csv")
print(df.shape)
df.head()


In [ ]:
# Load per-role models trained by Stage 2A
from pipeline.price_models import load_price_models, predict_price

mean_models, std_models = load_price_models(MODELS_DIR, tuple(CFG["roles"]["major"]))
print("Loaded models for roles:", list(mean_models))


In [ ]:
# --- D2: production inference ------------------------------------------------
price_pred, std_pred = np.empty(len(df)), np.empty(len(df))
for role in CFG["roles"]["major"]:
    m = df["Role"] == role
    p, s = predict_price(mean_models, std_models, df.loc[m, "FVM"], role,
                         std_min=CFG["inference"]["std_min"])
    price_pred[m.to_numpy()], std_pred[m.to_numpy()] = p, s

df["expprice"] = np.maximum(np.round(price_pred), CFG["inference"]["price_min"]).astype(int)
df["expstd"] = np.maximum(np.round(std_pred), CFG["inference"]["std_min"]).astype(int)
print(df.groupby("Role")[["expprice", "expstd"]]
      .agg(["count", "min", "max", "mean"]).round(1).to_string())


## Residual Correction Experiment (D3)

The FVM-only model cannot see expected performance. Players whose `ExpectedMf` exceeds
their `(role, FVM-band)` median may be undervalued. This cell compares correction
variants against **observed prices from the most recent auction** (stored in
`data/raw/{SEASON}`). It is an experiment: the production path (D4) uses the uncorrected
model unless a variant clearly wins and is enabled in config.


In [ ]:
# --- D3: residual-correction variants ----------------------------------------
from pipeline.auction_loader import load_auctions, prev_season

obs_season = prev_season(SEASON)
auctions = load_auctions(SEASON, rose_dirs=("current",),
                         rose_format=CFG["training"]["rose_format"])
auctions["name_key"] = auctions["name"].str.strip().str.lower()
obs = (auctions[auctions["auction_season"] == obs_season]
       .groupby(["role", "name_key"], as_index=False)["price"].mean()
       .rename(columns={"price": "obs_price"}))
eval_df = (
    df[["Role", "Name", "FVM", "ExpectedMf", "expprice"]]
    .assign(name_key=lambda d: d["Name"].str.strip().str.lower())
    .rename(columns={"Role": "role"})
    .merge(obs, on=["role", "name_key"], how="inner")
)
print(f"Players with observed price: {len(eval_df)}")

# ExpectedMf reference: median per (role, FVM band)
eval_df["band"] = pd.cut(eval_df["FVM"], [0, 50, 100, 200, 10 ** 6],
                         labels=["low", "mid", "high", "top"])
med = eval_df.groupby(["role", "band"], observed=True)["ExpectedMf"].median()
eval_df["mf_median"] = [med.get((r, b)) for r, b in zip(eval_df["role"], eval_df["band"])]

rw = CFG["inference"]["residual_weight"]
base = eval_df["expprice"].to_numpy(float)
dev = (eval_df["ExpectedMf"] - eval_df["mf_median"]).to_numpy(float)
variants = {
    "v0_none": base,
    "v1_additive": base + rw * dev * 10,
    "v2_multiplicative": base * (1 + rw * (eval_df["ExpectedMf"] / eval_df["mf_median"] - 1)),
    "v3_capped_additive": base + np.clip(rw * dev * 10, -5, 5),
}
obs_y = eval_df["obs_price"].to_numpy(float)
rows = [{"variant": k,
         "mae": round(float(np.abs(v - obs_y).mean()), 2),
         "rmse": round(float(np.sqrt(((v - obs_y) ** 2).mean())), 2)}
        for k, v in variants.items()]
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# --- D4: BE-ready export -------------------------------------------------------
out = df[["Id", "Role", "Role_M", "Name", "Squad", "Price", "Age", "MyRating",
          "Mate", "Regularness", "FVM", "ExpectedMf", "expprice", "expstd"]].copy()
out.columns = ["id", "role", "role_m", "name", "squad", "price", "age", "myrating",
               "mate", "regularness", "fvm", "expmf", "expprice", "expstd"]
Path(FINAL_DIR).mkdir(parents=True, exist_ok=True)
out.to_csv(Path(FINAL_DIR) / "players.csv", index=False)
print(f"Saved {FINAL_DIR}/players.csv ({len(out)} rows)")
out.head()


In [ ]:
# --- D5: validation -------------------------------------------------------------
BE_COLUMNS = ["id", "role", "role_m", "name", "squad", "price", "age", "myrating",
              "mate", "regularness", "fvm", "expmf", "expprice", "expstd"]
checks = {
    "columns_exact": list(out.columns) == BE_COLUMNS,
    "no_duplicate_id": bool(out["id"].is_unique),
    "expprice_positive": bool((out["expprice"] > 0).all()),
    "expstd_positive": bool((out["expstd"] > 0).all()),
    "myrating_range": bool(out["myrating"].between(1, 5).all()),
}
assert all(checks.values()), f"BE contract check failed: {checks}"
print(checks)
print("\nTop 10 by FVM:")
print(out.nlargest(10, "fvm")[["name", "role", "fvm", "price", "expprice", "expstd"]]
      .to_string(index=False))

report = {"season": SEASON, "rows": int(len(out)), "checks": checks,
          "expprice": [int(out["expprice"].min()), int(out["expprice"].max())],
          "expstd": [int(out["expstd"].min()), int(out["expstd"].max())]}
out_path = Path(INTER_DIR) / "validation_stage2b.json"
out_path.write_text(json.dumps(report, indent=2))
print(f"\nWritten {out_path}")
